# RoDiLoCo — one-click reproduction (Kaggle / Colab)

Run **Run All** (▶▶). This clones the repo, verifies the GPU and dataset, runs the full
**reproduce → attack → defend** campaign (~45 min on a T4), and renders the three headline
plots. Single seed, reduced scale — enough to show every phenomenon cleanly.

### Before you run
1. **Settings → Accelerator → GPU T4** (x1 or x2), **Internet → On**.
2. Set **`REPO_URL`** in the first code cell to your GitHub repo.
3. Make sure you've **pushed your latest local changes** (the notebook clones from GitHub).

This notebook is defensive: it **stops loudly** if the GPU is off, the dataset is missing, or
the repo is stale — so it can never silently fall back to CPU or a toy corpus.


## 1 · Get the code

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/MTahaB/RoDiLoCo.git"   

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO_DIR = os.path.join(WORK, "RoDiLoCo")
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--no-edit"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

# guard: refuse to run with a stale repo (missing the trust-weighted median-norm fix)
assert "median()" in open("src/rodiloco/aggregators.py").read(), \
    "This clone is missing the latest fix. Push your local changes to GitHub and re-run."
print("working dir:", os.getcwd(), "| repo is up to date")

## 2 · Dependencies + GPU check

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "pyyaml", "datasets", "matplotlib", "tqdm"], check=True)
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "GPU is NOT enabled. Stop, set Settings -> Accelerator -> GPU T4, then Run All again "
    "(this campaign on CPU would take ~20 hours)."
)
print("GPU:", torch.cuda.get_device_name(0))

## 3 · Dataset (TinyStories) + check

In [ ]:
DATA = "data/tinystories_train.txt"
if not os.path.exists(DATA) or os.path.getsize(DATA) < 1_000_000:
    subprocess.run([sys.executable, "scripts/prepare_data.py",
                    "--out", DATA, "--max-chars", "8000000"], check=True)
size = os.path.getsize(DATA)
assert size > 1_000_000, "TinyStories download failed - check Internet is On, then rerun."
print(f"dataset ready: {DATA}  ({size/1e6:.1f} MB)")

## 4 · Experiment configuration

One config, used for every run — so all results are directly comparable. `text_path` is set
explicitly, so the run can never silently fall back to the toy corpus.

In [ ]:
import yaml
from rodiloco.diloco import run_diloco

SEED = 0
CONFIG = dict(
    seed=SEED, device="auto", amp=True,
    text_path="data/tinystories_train.txt",
    seq_len=96, batch_size=24, n_workers=8, iid=True,
    H=50, outer_rounds=15,
    inner_lr=1e-3, outer_lr=0.7, outer_momentum=0.9, weight_decay=0.1, grad_clip=1.0,
    attack="sign_flip", attack_lam=10.0,
    eval_every=3, eval_batches=15,
    model=dict(d_model=192, n_layers=4, n_heads=6, max_seq_len=96),
)

def run(out, **over):
    return run_diloco({**CONFIG, "out_dir": out, **over})

print("config ready | ~%d inner steps/run" % (CONFIG["outer_rounds"]*CONFIG["n_workers"]*CONFIG["H"]))

## 5 · The full campaign  (~45 min)

- **Plot #1 — fragility:** naive `mean` under sign-flip, sweeping the byzantine fraction f.
- **Plots #2 & #3 — defenses:** every aggregator across f (transfer + robustness tax).

Progress prints per run; leave it and check back.

In [ ]:
import time
t0 = time.time()

# Plot #1 - fragility of the mean
for f in [0, 1, 2, 3]:
    run(f"outputs/p3_sign_flip_f{f}", aggregator="mean", n_byzantine=f)

# Plots #2 & #3 - defense comparison
DEFENSES = {
    "mean":             {},
    "trimmed_mean":     {"trim": 1},
    "krum":             {"n_byzantine": 2},
    "geometric_median": {},
    "centered_clipping":{"tau": 1.0},
    "trust_weighted":   {"beta": 0.9, "tau": 0.1},
}
for agg, kw in DEFENSES.items():
    for f in [0, 1, 2]:
        r = run(f"outputs/p4_{agg}_f{f}", aggregator=agg, aggregator_kwargs=kw, n_byzantine=f)
        print(f"{agg:18s} f={f}  final_ppl={r['final_ppl']:.2f}")

print(f"\nALL RUNS DONE in {(time.time()-t0)/60:.1f} min")

## 6 · The three headline plots

In [ ]:
for kind, pat, dst in [
    ("fragility", "outputs/p3_*",    "results/plot1_fragility.png"),
    ("defense",   "outputs/p4_*",    "results/plot2_defense.png"),
    ("tax",       "outputs/p4_*_f0", "results/plot3_tax.png"),
]:
    subprocess.run([sys.executable, "scripts/plot_results.py", kind, pat, dst], check=True)

from IPython.display import Image, display
for p in ["results/plot1_fragility.png", "results/plot2_defense.png", "results/plot3_tax.png"]:
    display(Image(p))

## 7 · Summary table

In [ ]:
import glob, json
rows = {}
for h in sorted(glob.glob("outputs/p4_*/history.json")):
    name = os.path.basename(os.path.dirname(h))          # p4_<agg>_f<f>
    agg, f = name[3:].rsplit("_f", 1)
    rows.setdefault(agg, {})[int(f)] = json.load(open(h))["final_ppl"]

print(f"{'defense':18s} {'f=0':>10s} {'f=1':>10s} {'f=2':>10s}")
for agg in ["mean","trimmed_mean","krum","geometric_median","centered_clipping","trust_weighted"]:
    r = rows.get(agg, {})
    def fmt(v): return "diverged" if v is None or v > 1e6 else f"{v:.2f}"
    print(f"{agg:18s} {fmt(r.get(0)):>10s} {fmt(r.get(1)):>10s} {fmt(r.get(2)):>10s}")

## 8 · Save your results

- **Download the 3 PNGs now:** files panel (right) → `results/` → download each. Do this
  before closing — `/kaggle/working` is wiped when the session resets.
- **Save Version** (top-right) to persist the whole run.

To replicate across seeds for the paper, change `SEED` in cell 4 and re-run (each seed ≈ 45 min).